Data Analysis on League of Legends 2026 Pro Tourneys

**Name(s)**: Khloe Wong, Noah Tung

**Website Link**: (your website link)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

Questions to answer: on the google doc

## Step 1: Introduction

In [2]:
# TODO
# will write the introduction section on the document and paste into notebook/website later

## Step 2: Data Cleaning and Exploratory Data Analysis

In [3]:
# Data Cleaning
# load the csv file in its entirety and display as a dataframe
lol_data = pd.read_csv("leagueesportsdata_2026.csv")

# grab the relevant columns for analysis for steps 1-4
# only take the TOTALS for each team so that we use overall -- at 15 since we're only interested in the team as a whole, not individual players
relevant_columns = ['goldat15', 'csat15', 'xpat15', 'golddiffat15', 'csdiffat15', 'xpdiffat15', 'killsat15', 'deathsat15', 'assistsat15', 'result']

# filter for COMPLETE in datacompleteness columns because SOME matches which are RANDOM do not have stats at 15.
# (MCAR) cannot fill in these NaN values since they are conditioned on the actual game that we do not have access to
cleaned_df = lol_data[(lol_data['position'] == 'team') & (lol_data['datacompleteness'] == 'complete')][relevant_columns]

# calculated to see how much data is incomplete, only 2%. that means we still have access to 98% of the data
completeness_prop = lol_data[lol_data['datacompleteness'] == 'complete'].shape[0] / lol_data.shape[0]

C:\Users\Khloe\AppData\Local\Temp\ipykernel_22792\4052382319.py:3: DtypeWarning: Columns (2,5) have mixed types. Specify dtype option on import or set low_memory=False.
  lol_data = pd.read_csv("leagueesportsdata_2026.csv")


In [4]:
# Univariate Analysis

# hist for goldat15
# observation: most teams average around 27k gold at 15 minutes. (center of data)
fig = px.histogram(cleaned_df, x = 'goldat15')
fig.show()

fig2 = px.histogram(cleaned_df, x = 'xpat15')
fig2.show()

# # hist for xpdiffat15
# # observation: teams average around 4-5 kills at 15 minutes
# fig2 = px.histogram(cleaned_df, x = 'killsat15')
# fig2.show()


In [5]:
# Bivariate Analysis
plot_df = cleaned_df.copy()
win_cleaned = cleaned_df[cleaned_df['result'] == 1]
loss_cleaned = cleaned_df[cleaned_df['result'] == 0]
# TODO: change the mapping to a legend/key***********************************
# Observation: Teams who lost had on average approx. 2k less gold at 15 minutes than teams that won.
# Code that matches above observation. (approx. 1732 gold)
mean_diff = win_cleaned['goldat15'].mean() - loss_cleaned['goldat15'].mean()
#hist1
plot_df['outcome'] = plot_df['result'].map({1: 'Win', 0: 'Loss'})
fig = px.histogram(
    plot_df,
    x='goldat15',
    color='outcome',
    barmode='overlay',
    opacity=0.6,
    nbins=100,
    title='Team Gold at 15 minutes: Wins vs. Losses',
    labels={'goldat15': 'Gold Per Team at 15 minutes', 'outcome': 'Result'},
)
fig.show()
# hist2
plot_df['outcome'] = plot_df['result'].map({1: 'Win', 0: 'Loss'})
fig = px.histogram(
    plot_df,
    x='csat15',
    color='outcome',
    barmode='overlay',
    opacity=0.6,
    nbins=100,
    title='Team CS at 15 minutes: Wins vs. Losses',
    labels={'csat15': 'CS Per Team at 15 minutes', 'outcome': 'Result'},
)
fig.show()
# around 22-23 CS lead for winning teams, but not significant enough to change the game (440 gold)
csmeandiff = win_cleaned['csat15'].mean() - loss_cleaned['csat15'].mean()

#scatterplot : the more kills you get, the more gold you get!
fig = px.scatter(cleaned_df, x = 'killsat15', y = 'goldat15')
fig.show()



In [6]:
# Interesting Aggregates
# Pivot table for statistics that we've graphed before and can see a difference between winning teams and losing teams in total
table = pd.pivot_table(cleaned_df, index = 'result', values = ['goldat15', 'csat15', 'xpat15', 'killsat15', 'assistsat15'], aggfunc= 'sum')
table

,assistsat15,csat15,goldat15,killsat15,xpat15
result,,,,,
0,60495.0,5224545.0,251297741.0,39277.0,311679142.0
1,90939.0,5443998.0,268164215.0,59208.0,325026284.0


## Step 3: Assessment of Missingness

In [7]:
# NMAR Analysis
# only NMAR is ban1 ban2 ban3 ban4 ban5 since it depends on the value itself. players can choose if 
# they would like to use or not use their ban, which is determined by the value itself and not any other column.

In [8]:
# Missingness Dependency
# SELECTED COLUMN: goldat15 column
# MISSINGNESS DOESN'T DEPEND ON: SIDE (blue/red)
    # Permutation Test
        # null: distribution of side when goldat15 is missing is 
            # the same as the distribution of side when goldat15 is not missing.
        # alt: distribution of side when goldat15 is missing is 
            # NOT the same as the distribution of side when goldat15 is not missing.
    # aiming to fail to reject the null
# MISSINGNESS DEPENDS ON: LEAGUE (LPL, LCK, etc.)
    # Permutation Test
        # null: distribution of league when goldat15 is missing is 
            # the same as the distribution of league when goldat15 is not missing.
        # alt: distribution of league when goldat15 is missing is
            # NOT the same as the distribution of league when goldat15 is not missing.
    # aiming to reject the null

In [9]:
team_df = lol_data[lol_data['position'] == 'team'].copy()
team_df['goldat15_missing'] = team_df['goldat15'].isna()

print('Rows:', len(team_df))
print('Missing goldat15:', team_df['goldat15_missing'].sum(),
      f"({team_df['goldat15_missing'].mean():.2%})")


# ---------------------------------------------------------------
# Test statistic: total variation distance (TVD) between the
# distribution of a categorical column when goldat15 is missing
# vs. when it is not missing.
# ---------------------------------------------------------------
def tvd(codes, missing, n_categories):
    dist_missing = np.bincount(codes[missing], minlength=n_categories) / missing.sum()
    dist_present = np.bincount(codes[~missing], minlength=n_categories) / (~missing).sum()
    return np.abs(dist_missing - dist_present).sum() / 2


def missingness_perm_test(data, col, n_reps=1000, seed=42):
    """Permutation test: does the distribution of `col` differ between
    rows where goldat15 is missing and rows where it is not?"""
    data = data.dropna(subset=[col])
    codes, uniques = pd.factorize(data[col])
    missing = data['goldat15_missing'].to_numpy()
    k = len(uniques)

    observed = tvd(codes, missing, k)

    rng = np.random.default_rng(seed)
    simulated = np.array([
        tvd(codes, rng.permutation(missing), k) for _ in range(n_reps)
    ])
    p_value = (simulated >= observed).mean()
    return observed, simulated, p_value


def plot_perm_test(simulated, observed, p_value, col):
    fig = px.histogram(
        x=simulated, nbins=50, histnorm='probability',
        title=f'Empirical distribution of TVD for {col} '
              f'(observed TVD = {observed:.4f}, p = {p_value:.4f})',
        labels={'x': f'TVD of {col} distributions (missing vs. not missing)'},
    )
    fig.add_vline(x=observed, line_color='red', line_width=3,
                  annotation_text='observed', annotation_position='top right')
    fig.update_layout(yaxis_title='Probability', showlegend=False)
    return fig


ALPHA = 0.05

# ---------------------------------------------------------------
# Test 1: goldat15 missingness vs. SIDE (expected: fail to reject)
# ---------------------------------------------------------------
obs_side, sim_side, p_side = missingness_perm_test(team_df, 'side')
print(f'\nSIDE   -> observed TVD = {obs_side:.4f}, p-value = {p_side:.4f}')
print('Reject null' if p_side < ALPHA else 'Fail to reject null')

fig_side = plot_perm_test(sim_side, obs_side, p_side, 'side')
fig_side.show()

# ---------------------------------------------------------------
# Test 2: goldat15 missingness vs. LEAGUE (expected: reject)
# ---------------------------------------------------------------
obs_league, sim_league, p_league = missingness_perm_test(team_df, 'league')
print(f'\nLEAGUE -> observed TVD = {obs_league:.4f}, p-value = {p_league:.4f}')
print('Reject null' if p_league < ALPHA else 'Fail to reject null')

fig_league_test = plot_perm_test(sim_league, obs_league, p_league, 'league')
fig_league_test.show()

# ---------------------------------------------------------------
# Plot: distribution of league when goldat15 is missing vs. not
# ---------------------------------------------------------------
league_dist = (
    team_df.groupby('goldat15_missing')['league']
    .value_counts(normalize=True)
    .rename('proportion')
    .reset_index()
)
league_dist['goldat15'] = league_dist['goldat15_missing'].map(
    {True: 'Missing', False: 'Not missing'}
)

# order leagues by their share among the missing rows
order = (league_dist[league_dist['goldat15'] == 'Missing']
         .sort_values('proportion', ascending=False)['league'].tolist())
order += [lg for lg in league_dist['league'].unique() if lg not in order]

fig_dist = px.bar(
    league_dist, x='league', y='proportion', color='goldat15',
    barmode='group', category_orders={'league': order},
    title='Distribution of league when goldat15 is missing vs. not missing',
)
fig_dist.show()

# Which leagues have the highest share of missing goldat15 rows?
by_league = (team_df.groupby('league')['goldat15_missing']
             .agg(['mean', 'sum', 'count'])
             .sort_values('mean', ascending=False))
print('\nLeagues with the most missing goldat15:')
print(by_league.head(8))

Rows: 19830
Missing goldat15: 348 (1.75%)

SIDE   -> observed TVD = 0.0000, p-value = 1.0000
Fail to reject null



LEAGUE -> observed TVD = 0.9346, p-value = 0.0000
Reject null



Leagues with the most missing goldat15:
            mean  sum  count
league                      
DCup    1.000000    6      6
WLRQ    1.000000   12     12
DCGI    0.450704   64    142
LPL     0.195846  264   1348
LEC     0.002611    2    766
CBLOL   0.000000    0    598
CD      0.000000    0    516
CCWS    0.000000    0     80


## Step 4: Hypothesis Testing

In [ ]:
# Hypothesis Testing
# null: the distribution of kills at 15 for winning teams that get first drake is the same as the team that didn't get first drake.
# alt: the distribution of kills at 15 for winning teams that get first drake is NOT the same as the team that didn't get first drake.
# test statistic: abs mean diff

## Step 5: Framing a Prediction Problem

In [ ]:
# Problem Identification
# in the google doc

## Step 6: Baseline Model

In [ ]:
# TODO

## Step 7: Final Model

In [ ]:
# TODO

## Step 8: Fairness Analysis

In [ ]:
# TODO